### Load data

In [1]:
import json

# load 1,000 pubmedqa questions (use json to keep abstract and ID attached)
with open("pubmedqa/data/ori_pqal.json") as f:
    data = json.load(f)

print(len(data))

1000


### Line up questions with their abstracts

In [2]:
# line up each question with its matching abstract and abstract id
abstract_ids = []
abstracts = []
questions = []

for abstract_id, entry in data.items():
    abstract_ids.append(abstract_id)
    abstracts.append(" ".join(entry["CONTEXTS"]))
    questions.append(entry["QUESTION"])

print(len(abstracts))

1000


### Sparse search (BM25)

In [3]:
import bm25s

# split each abstract into words (also drops filler words)
abstract_tokens = bm25s.tokenize(abstracts, stopwords="en")

# create keyword search index for all abstracts
bm25 = bm25s.BM25()
bm25.index(abstract_tokens)

/Users/andrew/Documents/Berkeley/MIDS/DS266/Final Project/DS266_RAG/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# try one question to see what bm25 finds
question = questions[0]
question_tokens = bm25s.tokenize(question, stopwords="en")

# retrieve top 5 results
top_positions, top_scores = bm25.retrieve(question_tokens, k=5)

print("question:", question)
print("correct abstract is at position: 0")
print("bm25's top 5 positions:", top_positions[0])
print("bm25's scores:", top_scores[0])

question: Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?
correct abstract is at position: 0
bm25's top 5 positions: [  0 265 111 244 450]
bm25's scores: [21.529465   9.112523   5.512696   4.3754864  4.285845 ]


### Dense search (PubMedBERT)

In [7]:
from txtai import Embeddings

# turn every abstract into numbers that capture its meaning
dense = Embeddings(path="neuml/pubmedbert-base-embeddings")
dense.index(abstracts)

W1029 11:53:54.636000 7736 .venv/lib/python3.14/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 18714.92it/s]


In [9]:
# try the same question with dense search
dense_results = dense.search(question, 5)

print("question:", question)
print("correct abstract is at position: 0")
print("dense's top 5 results:", dense_results)

question: Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?
correct abstract is at position: 0
dense's top 5 results: [(0, 0.6894745230674744), (265, 0.3121076822280884), (76, 0.18515199422836304), (199, 0.16055384278297424), (751, 0.15123626589775085)]
